In [ ]:
import pandas as pd
import numpy as np

## Explicação do dataset:<br>
A base de dados em questão foi retirada do dados.gov e traz informações sobre mortalidade, temos dados dos óbitos ocorridos até 31/05/2026.
link da base de dados: Mortalidade Geral 2026 1ª prévia em https://dados.gov.br/dados/conjuntos-dados/sim-1979-2019 <br>

#### Colunas:
**contador: identificador do registro<br>**
**TIPOOBITO: se é um óbito fetal ou não<br>**
**DTOBITO: data em que ocorreu o óbito<br>**
**DTNASC: data de nascimento<br>**
**IDADE: campo que mostra em minutos, horas, dias ou anos<br>**
**SEXO: M,1 - masculino; F,2 - feminino; I,0,9 - ignorado<br>**
**RACACOR: 1- branca; 2-preta; 3-amarela; 4-parda; 5-indigena<br>**
**ESTCIV: 1– Solteiro; 2 – Casado; 3 – Viúvo; 4 – Separado judicialmente/divorciado; 5 – União estável; 9 – Ignorado<br>**
**ESC2010:0 – Sem escolaridade; 1 – Fundamental I (1ª a 4ª série); 2 – Fundamental II (5ª a 8ª série); 3 – Médio (antigo 2º Grau); 4 – Superior incompleto; 5 – Superior completo; 9 – Ignorado<br>**
**OCUP: só preenchido se o cammpo idade for maior que 5 anos. Codigo de acordo com a CBO 2002<br>**
**CODMUNRES: codigo de municipio de residencia do falecido<br>**
**CODMUNOCOR: codigo de municipio onde ocorreu o óbito<br>**
**CAUSABAS: causa básica da morte (CID)<br>**
**ACIDTRAB: 1- sim; 2-não; 9-ignorado <br>**

#### Perguntas que serão respondidas:

In [ ]:
# 1. Define a lista de colunas que serão importadas
colunas_desejadas = [
    'contador',
    'TIPOBITO',
    'DTOBITO',
    'DTNASC',
    'IDADE',
    'SEXO',
    'RACACOR',
    'ESTCIV',
    'ESC2010',
    'OCUP',
    'CODMUNRES',
    'CODMUNOCOR',
    'CAUSABAS',
    'ACIDTRAB'
]

# 2. Defina os tipos de dados das colunas
tipos_dados = {
    'TIPOBITO': str,
    'IDADE': int,       # Mantém como texto para não perder os zeros à esquerda da codificação do DATASUS
    'SEXO': int,
    'RACACOR': str,
    'ESTCIV': str,
    'ESC2010': str,
    'OCUP': str,
    'CODMUNRES': int,
    'CODMUNOCOR': str,
    'CAUSABAS': str,   # Garante que códigos CID mistos não gerem alertas
}

# 3. Importa o arquivo CSV de forma otimizada
mortalidade = pd.read_csv(
    'Mortalidade_Geral_2026.csv',
    sep=';',
    usecols=colunas_desejadas,
    dtype=tipos_dados,
    parse_dates=['DTOBITO', 'DTNASC'],
    date_format='%d/%m/%Y'
)

# 4. Converte DTOBITO e DTNASC (formato: ddmmYYYY) para uma data real do Pandas (formato: YYYY-mm-dd)
mortalidade['DTOBITO'] = pd.to_datetime(mortalidade['DTOBITO'], format='%d%m%Y', errors='coerce')
mortalidade['DTNASC'] = pd.to_datetime(mortalidade['DTNASC'], format='%d%m%Y', errors='coerce')

In [ ]:
mortalidade.head()

In [ ]:
mortalidade.info()

In [ ]:
mortalidade.describe()

In [ ]:
municipios = pd.read_excel('MUNICIPIOS.xlsx')

In [ ]:
municipios.head()

In [ ]:
municipios.info()

**Abaixo feito um merge com a tabela de municipios para poder analisar os dados de cada municipio**

In [ ]:
mortalidade = mortalidade.merge(municipios, how='left')

In [ ]:
mortalidade.head()

In [ ]:
mortalidade.info()

In [ ]:
cid = pd.read_excel('CIDs_com_descricao.xlsx')

In [ ]:
cid.head()

**feito um merge com a tabela de cids, para poder identificar as maiores causas de morte em 2026**

In [ ]:
mortalidade = mortalidade.merge(cid, how='left', left_on='CAUSABAS', right_on='CAUSABAS')

In [ ]:
mortalidade.head()

In [ ]:
mortalidade.info()

In [ ]:
mortalidade['CAUSABAS'].isna().sum()

In [ ]:
mortalidade['DESCRICAO_CID'].isna().sum()

**cids que não foram localizados na tabela oficial**

In [ ]:
mortalidade.loc[mortalidade['DESCRICAO_CID'].isna(), ['CAUSABAS', 'DESCRICAO_CID']]

In [ ]:
cids_faltantes = pd.DataFrame(cid.loc[cid['DESCRICAO_CID'].isna(), ['CAUSABAS', 'DESCRICAO_CID']])

In [ ]:
cids_faltantes.to_csv('CIDs_faltantes.csv', index=False)

In [ ]:
cid.loc[cid['CAUSABAS']=='A090', ['CAUSABAS', 'DESCRICAO_CID']]

In [ ]:
diag = pd.DataFrame({
    'faltantes': mortalidade.isna().sum(),
    'pct': (mortalidade.isna().mean() * 100).round(2),
    'distintos': mortalidade.nunique(),
})
diag[diag['faltantes'] > 0].sort_values('faltantes', ascending=False)

In [ ]:
print("linhas com algum faltante:", mortalidade.isna().any(axis=1).sum())
print("linhas completas         :", mortalidade.notna().all(axis=1).sum())

**não temos nenhuma linha dupllicada no dataset, como podemos verificar abaixo**

In [ ]:
print("linhas totalmente duplicadas:", mortalidade.duplicated().sum())
mortalidade[mortalidade.duplicated(keep=False)].sort_values('contador')

**10 maiores causas de morte em 2026:**

In [ ]:
mortalidade['DESCRICAO_CID'].value_counts().head(10)

**10 maiores municipios em quantidade de mortes em 2026:**

In [ ]:
mortalidade['MUNICIPIO'].value_counts().head(10)

In [ ]:
mortalidade['ACIDTRAB'].isna().sum()

In [ ]:
acidentes = mortalidade[mortalidade['ACIDTRAB']== 1.0]

In [ ]:
acidentes.head()

**10 maiores causas de morte por acidentes de trabalho em 2026:**

In [ ]:
acidentes.groupby(['CAUSABAS','DESCRICAO_CID' ]).size().sort_values(ascending=False).head(10)